# MedVision Agent â€” Lung Nodule U-Net (LUNA16)

> âš ï¸ **This is a research prototype and not approved for clinical diagnosis.**
> Use public datasets only and respect their licenses.

**Goal:** train a tiny 2-D U-Net on axial chest-CT slices to segment lung
nodules/tumors, then export `lung_unet.onnx`.

- Dataset options:
  - **LUNA16** (subset of LIDC-IDRI): `.mhd/.raw` volumes + `annotations.csv`
    (world-coords nodule centers + diameter). https://luna16.grand-challenge.org/
  - **MSD Task06 Lung** works too with a small adapter.
- Nodule masks are synthesized as circles from annotation geometry (the classic
  LUNA16 weak-label approach) â€” coarse but trainable.
- HU windowing: clip to `[-1000, 400]` then scale to `[0, 1]`.

In [ ]:
%pip install -q SimpleITK onnx onnxruntime matplotlib tqdm pandas
# torch + torchvision preinstalled on Colab/Kaggle GPUs

In [ ]:
import json, math, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

CONFIG = {
    'SIZE': 256,
    'BATCH': 16,
    'EPOCHS': 25,
    'LR': 1e-3,
    'MAX_SERIES': 30,       # teaching subset; raise for real experiments
    'HU_CLIP': (-1000.0, 400.0),
    'SEED': 42,
}
OUT_DIR = Path('outputs'); OUT_DIR.mkdir(exist_ok=True)
print('device:', DEVICE)

## 1 Â· Locate LUNA16 data

Expected layout:
```
DATA_ROOT/
  subset0/*.mhd *.raw
  ...
  annotations.csv   (seriesuid, coordX, coordY, coordZ, diameter_mm)
```

In [ ]:
# TODO(student): download LUNA16 (respect the license) and set DATA_ROOT.
CANDIDATES = ['/kaggle/input/luna16', '/content/LUNA16', './data/luna16']
DATA_ROOT = next((p for p in CANDIDATES if Path(p).exists()), None)
assert DATA_ROOT, 'Set DATA_ROOT to your LUNA16 folder containing subsets*/ and annotations.csv.'

import pandas as pd
ann = pd.read_csv(next(Path(DATA_ROOT).rglob('annotations.csv')))
ann.columns = ['seriesuid', 'x', 'y', 'z', 'd']  # normalize header casing

series_paths = {}
for mhd in sorted(Path(DATA_ROOT).rglob('*.mhd')):
    series_paths[mhd.stem] = mhd  # stem == seriesuid for LUNA16
keep = [s for s in ann.seriesuid.unique() if s in series_paths][: CONFIG['MAX_SERIES']]
ann = ann[ann.seriesuid.isin(keep)]
print(f'{len(keep)} series | {len(ann)} annotated nodules')

In [ ]:
import SimpleITK as sitk

HU_LO, HU_HI = CONFIG['HU_CLIP']

def window_hu(slice_hu):
    x = np.clip(slice_hu, HU_LO, HU_HI).astype(np.float32)
    return (x - HU_LO) / (HU_HI - HU_LO)

class LungSliceDataset(Dataset):
    '''Axial CT slices with circle masks synthesized from LUNA16 annotations.'''

    def __init__(self, series_ids, size=256, augment=False, neg_per_pos=1.0):
        self.size, self.augment = size, augment
        self._cache = {}
        self.index = []            # (series_id, z, [(cy, cx, r_px), ...])
        nod_by_series = {s: ann[ann.seriesuid == s] for s in set(series_ids)}
        per_series_pos = {}
        for sid in series_ids:
            sitk_img, arr_z = self._load(sid)
            origin, spacing = sitk_img.GetOrigin(), sitk_img.GetSpacing()  # mm
            nz = arr_z.shape[0]
            slices_nod = {}
            for _, row in nod_by_series[sid].iterrows():
                vz = (row.z - origin[2]) / spacing[2]
                r_vox = max(row.d / 2.0 / spacing[0], 2.0)
                z_span = int(math.ceil(r_vox * spacing[0] / max(spacing[2], 1e-6)))
                for dz in range(-z_span, z_span + 1):
                    zz = int(round(vz)) + dz
                    if not (0 <= zz < nz):
                        continue
                    chord = r_vox**2 - (dz * spacing[2] / spacing[0])**2
                    if chord <= 0:
                        continue
                    r_slice = math.sqrt(chord)
                    cy = (row.y - origin[1]) / spacing[1]
                    cx = (row.x - origin[0]) / spacing[0]
                    slices_nod.setdefault(zz, []).append((cy, cx, r_slice))
            pos = sorted(slices_nod.keys())
            rng = random.Random((hash(sid) ^ 0x1056) & 0xFFFF)
            n_neg = min(int(len(pos) * neg_per_pos), nz - len(pos))
            neg = rng.sample([z for z in range(nz) if z not in slices_nod], n_neg)
            for z in pos:
                self.index.append((sid, z, slices_nod[z]))
            for z in neg:
                self.index.append((sid, z, []))
            per_series_pos[sid] = len(pos)
        self.series_ids = list(per_series_pos.keys())

    def _load(self, sid):
        if sid not in self._cache:
            img = sitk.ReadImage(str(series_paths[sid]))
            self._cache[sid] = (img, sitk.GetArrayFromImage(img).astype(np.float32))
        return self._cache[sid]

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        sid, z, nods = self.index[i]
        _, vol = self._load(sid)
        h, w = vol.shape[1], vol.shape[2]
        x = window_hu(vol[z])
        y = np.zeros((h, w), dtype=np.float32)
        yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
        for cy, cx, r in nods:
            y[((yy - cy) ** 2 + (xx - cx) ** 2) <= r ** 2] = 1.0

        if self.augment:
            if random.random() < 0.5:
                x, y = x[:, ::-1].copy(), y[:, ::-1].copy()
            k = random.randint(0, 3)
            if k:
                x, y = np.rot90(x, k).copy(), np.rot90(y, k).copy()
            x = x * np.random.uniform(0.92, 1.08)

        rs = (self.size, self.size)
        x = np.asarray(Image.fromarray((x * 255).astype(np.uint8), 'L').resize(rs, Image.BILINEAR), np.float32) / 255.0
        y = np.asarray(Image.fromarray((y * 255).astype(np.uint8), 'L').resize(rs, Image.NEAREST), np.float32)
        return torch.from_numpy(x[None]).float(), (y > 0.5).float()[None]

split = int(len(keep) * 0.85)
train_ds = LungSliceDataset(keep[:split], CONFIG['SIZE'], augment=True)
val_ds   = LungSliceDataset(keep[split:], CONFIG['SIZE'], augment=False)
train_loader = DataLoader(train_ds, batch_size=CONFIG['BATCH'], shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=CONFIG['BATCH'], shuffle=False, num_workers=2, pin_memory=True)
pos_share = sum(1 for _,_,n in train_ds.index if n) / max(len(train_ds), 1)
print(f'train slices {len(train_ds)} ({pos_share:.0%} nodule-bearing) | val series {len(val_ds.series_ids)}')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for j in range(4):
    xi = next(i for i in random.sample(range(len(train_ds)), len(train_ds)) if train_ds.index[i][2])
    x, y = train_ds[xi]
    axes[0, j].imshow(x[0], cmap='gray'); axes[0, j].axis('off')
    axes[1, j].imshow(x[0], cmap='gray'); axes[1, j].imshow(y[0], alpha=0.4, cmap='Reds'); axes[1, j].axis('off')
plt.suptitle('CT slice + synthesized circle mask'); plt.tight_layout(); plt.show()

## 2 Â· Model, loss, metrics â€” shared MiniUNet

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class MiniUNet(nn.Module):
    def __init__(self, base=16):
        super().__init__()
        self.d1, self.d2, self.d3 = DoubleConv(1, base), DoubleConv(base, base*2), DoubleConv(base*2, base*4)
        self.pool = nn.MaxPool2d(2)
        self.u2, self.u1 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2), nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.c2, self.c1 = DoubleConv(base*4, base*2), DoubleConv(base*2, base)
        self.out = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        s1 = self.d1(x); s2 = self.d2(self.pool(s1)); b = self.d3(self.pool(s2))
        u = self.c2(torch.cat([self.u2(b), s2], dim=1))
        u = self.c1(torch.cat([self.u1(u), s1], dim=1))
        return self.out(u)

model = MiniUNet().to(DEVICE)
print(f'{sum(p.numel() for p in model.parameters())/1e6:.2f} M parameters')

class DiceBCELoss(nn.Module):
    def __init__(self, dice_weight=0.5, smooth=1.0):
        super().__init__()
        self.bce, self.w, self.smooth = nn.BCEWithLogitsLoss(), dice_weight, smooth
    def forward(self, logits, target):
        bce = self.bce(logits, target)
        prob = torch.sigmoid(logits)
        inter = (prob * target).sum(dim=(2, 3))
        union = prob.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        dice = 1 - ((2 * inter + self.smooth) / (union + self.smooth)).mean()
        return bce + self.w * dice

@torch.no_grad()
def segmentation_metrics(logits, target, thr=0.5, eps=1e-7):
    pred = (torch.sigmoid(logits) > thr).float()
    tp = (pred * target).sum(); fp = (pred * (1 - target)).sum(); fn = ((1 - pred) * target).sum()
    return {
        'dice': float((2 * tp + eps) / (2 * tp + fp + fn + eps)),
        'iou': float((tp + eps) / (tp + fp + fn + eps)),
        'precision': float((tp + eps) / (tp + fp + eps)),
        'recall': float((tp + eps) / (tp + fn + eps)),
    }

criterion = DiceBCELoss(dice_weight=0.7)  # tiny lesions â†’ lean on Dice harder
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['LR'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))

In [ ]:
def train_one_epoch(loader):
    model.train(); total = 0.0
    for img, msk in loader:
        img, msk = img.to(DEVICE), msk.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
            loss = criterion(model(img), msk)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        total += float(loss) * img.size(0)
    return total / max(1, len(loader.dataset))

@torch.no_grad()
def evaluate(loader):
    model.eval(); total = 0.0; agg = {'dice': [], 'iou': [], 'precision': [], 'recall': []}
    for img, msk in loader:
        img, msk = img.to(DEVICE), msk.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
            logits = model(img)
        total += float(criterion(logits.float(), msk)) * img.size(0)
        for k, v in segmentation_metrics(logits.float(), msk).items():
            agg[k].append(v)
    out = {k: sum(v) / len(v) for k, v in agg.items()}
    out['loss'] = total / max(1, len(loader.dataset))
    return out

history = {'train_loss': [], 'val_loss': [], 'val_dice': [], 'val_iou': []}
best_dice, best_path = -1.0, OUT_DIR / 'lung_unet_best.pth'

for epoch in range(1, CONFIG['EPOCHS'] + 1):
    tl = train_one_epoch(train_loader)
    ev = evaluate(val_loader)
    scheduler.step(ev['dice'])
    history['train_loss'].append(tl); history['val_loss'].append(ev['loss'])
    history['val_dice'].append(ev['dice']); history['val_iou'].append(ev['iou'])
    star = ''
    if ev['dice'] > best_dice:
        best_dice = ev['dice']; torch.save(model.state_dict(), best_path); star = '  â† saved'
    print(f"epoch {epoch:02d} | train_loss {tl:.4f} | val_loss {ev['loss']:.4f} | "
          f"dice {ev['dice']:.4f} | iou {ev['iou']:.4f}{star}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history['train_loss'], label='train'); ax[0].plot(history['val_loss'], label='val')
ax[0].set_title('Dice+BCE loss'); ax[0].legend()
ax[1].plot(history['val_dice'], label='dice'); ax[1].plot(history['val_iou'], label='iou')
ax[1].set_title('validation overlap metrics'); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
final = evaluate(val_loader)
print(json.dumps(final, indent=2))

onnx_path = OUT_DIR / 'lung_unet.onnx'
dummy = torch.randn(1, 1, CONFIG['SIZE'], CONFIG['SIZE'])
model.eval().cpu()
torch.onnx.export(
    model, dummy, str(onnx_path), opset_version=17,
    input_names=['input'], output_names=['logits'],
    dynamic_axes={'input': {0: 'batch', 2: 'h', 3: 'w'}, 'logits': {0: 'batch', 2: 'h', 3: 'w'}},
)

import onnx, onnxruntime as ort
onnx.checker.check_model(onnx.load(str(onnx_path)))
sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
with torch.no_grad():
    ref = torch.sigmoid(model(dummy)).numpy()
raw = sess.run(None, {'input': dummy.numpy()})[0]
prob = raw if (raw.min() >= 0 and raw.max() <= 1) else 1 / (1 + np.exp(-raw))
diff = float(np.abs(prob - ref).max())
print(f'ONNX vs PyTorch max |Î”|: {diff:.2e}')
assert diff < 1e-3, 'export parity check failed'
print(f'Done â†’ copy {onnx_path} to inference-service/models/lung_unet.onnx')

## Next steps & honest caveats

- Circle masks from annotations are **weak labels**; expect blob-like outputs
  rather than LIDC-quality contours. Multi-rater LIDC masks improve realism.
- Report detection-style recall@IoU thresholds separately from segmentation Dice.
- Copy `outputs/lung_unet.onnx` â†’ `inference-service/models/`, then set
  `MOCK_PREDICTIONS=0`.

> **This is a research prototype and not approved for clinical diagnosis.**